In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class InitialBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(InitialBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels - 3, kernel_size=3, stride=2, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        main = self.conv(x)
        side = self.pool(x)
        x = torch.cat((main, side), dim=1)
        x = self.batch_norm(x)
        return F.relu(x)

class Bottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding, dilation=1, asymmetric=False, downsample=False, dropout_prob=0.1):
        super(Bottleneck, self).__init__()
        self.downsample = downsample
        self.asymmetric = asymmetric

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.batch_norm1 = nn.BatchNorm2d(out_channels)

        if downsample:
            self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, stride=2, padding=padding, dilation=dilation, bias=False)
        elif asymmetric:
            self.conv2 = nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=(kernel_size, 1), padding=(padding, 0), dilation=dilation, bias=False),
                nn.Conv2d(out_channels, out_channels, kernel_size=(1, kernel_size), padding=(0, padding), dilation=dilation, bias=False)
            )
        else:
            self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=padding, dilation=dilation, bias=False)

        self.batch_norm2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False)
        self.batch_norm3 = nn.BatchNorm2d(out_channels)

        self.dropout = nn.Dropout2d(p=dropout_prob)

        self.downsample_layer = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if downsample else None

    def forward(self, x):
        main = self.conv1(x)
        main = self.batch_norm1(main)
        main = F.relu(main)

        main = self.conv2(main)
        main = self.batch_norm2(main)
        main = F.relu(main)

        main = self.conv3(main)
        main = self.batch_norm3(main)
        main = self.dropout(main)

        if self.downsample:
            x = self.downsample_layer(x)

        return F.relu(main + x)

class ENet(nn.Module):
    def __init__(self, num_classes):
        super(ENet, self).__init__()
        self.initial_block = InitialBlock(in_channels=3, out_channels=16)

        self.bottleneck1_0 = Bottleneck(in_channels=16, out_channels=64, kernel_size=3, padding=1, downsample=True)
        self.bottleneck1_1 = Bottleneck(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bottleneck1_2 = Bottleneck(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bottleneck1_3 = Bottleneck(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bottleneck1_4 = Bottleneck(in_channels=64, out_channels=64, kernel_size=3, padding=1)

        self.bottleneck2_0 = Bottleneck(in_channels=64, out_channels=128, kernel_size=3, padding=1, downsample=True)
        self.bottleneck2_1 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck2_2 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=2, dilation=2)
        self.bottleneck2_3 = Bottleneck(in_channels=128, out_channels=128, kernel_size=5, padding=2, asymmetric=True)
        self.bottleneck2_4 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=4, dilation=4)
        self.bottleneck2_5 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck2_6 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=2, dilation=2)
        self.bottleneck2_7 = Bottleneck(in_channels=128, out_channels=128, kernel_size=5, padding=2, asymmetric=True)
        self.bottleneck2_8 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=4, dilation=4)

        self.bottleneck3_0 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck3_1 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck3_2 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck3_3 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bottleneck3_4 = Bottleneck(in_channels=128, out_channels=128, kernel_size=3, padding=1)

        self.fullconv = nn.ConvTranspose2d(in_channels=128, out_channels=num_classes, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)

    def forward(self, x):
        x = self.initial_block(x)
        x = self.bottleneck1_0(x)
        x = self.bottleneck1_1(x)
        x = self.bottleneck1_2(x)
        x = self.bottleneck1_3(x)
        x = self.bottleneck1_4(x)

        x = self.bottleneck2_0(x)
        x = self.bottleneck2_1(x)
        x = self.bottleneck2_2(x)
        x = self.bottleneck2_3(x)
        x = self.bottleneck2_4(x)
        x = self.bottleneck2_5(x)
        x = self.bottleneck2_6(x)
        x = self.bottleneck2_7(x)
        x = self.bottleneck2_8(x)

        x = self.bottleneck3_0(x)
        x = self.bottleneck3_1(x)
        x = self.bottleneck3_2(x)
        x = self.bottleneck3_3(x)
        x = self.bottleneck3_4(x)

        x = self.fullconv(x)

        return F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)

if __name__ == "__main__":
    # Defining the model with the number of output classes
    num_classes = 4  #The class of the dataset is 4 so class is set equals to 4
    model = ENet(num_classes)

    # Creating a random input tensor simulating a batch of images
    input_tensor = torch.randn(1, 3, 512, 512)  # Batch size of 1, 3 color channels, 512x512 image size

    # Forward pass through the model
    output = model(input_tensor)
    print(output.shape)  # Output shape should be (batch_size, num_classes, height, width)


torch.Size([1, 4, 256, 256])


In [ ]:
#mounting my google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import os
import cv2
import torchvision.transforms as transforms

class LaneDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = os.listdir(image_dir)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        label_path = os.path.join(self.label_dir, self.image_files[idx].replace('.jpg', '.png'))

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            image = self.transform(image)
            label = self.transform(label)

        return image, label

if __name__ == "__main__":
    image_dir = "/content/drive/My Drive/lane_data"
    label_dir = "/content/drive/My Drive/lane_labels"

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((512, 512))
    ])

    # Loading the dataset
    full_dataset = LaneDataset(image_dir, label_dir, transform=transform)

    # Defining the split ratios
    train_size = int(0.7 * len(full_dataset))  # 70% for training
    test_size = len(full_dataset) - train_size  # 30% for testing

    # Splitting the dataset
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    # Creating DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, pin_memory=True)

    # Checking the datasets
    for images, labels in train_loader:
        print("Train batch:", images.shape, labels.shape)
        break

    for images, labels in test_loader:
        print("Test batch:", images.shape, labels.shape)
        break


Train batch: torch.Size([64, 3, 512, 512]) torch.Size([64, 1, 512, 512])
Test batch: torch.Size([64, 3, 512, 512]) torch.Size([64, 1, 512, 512])


In [ ]:
#model training
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Defining the model, loss function, and optimizer
model = ENet(num_classes=4)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Moving the model to the GPU because it is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Defining the training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
      images = images.to(device)
      labels = labels.to(device)
      if len(labels.shape) == 4:
        labels = torch.squeeze(labels, dim=1)

      labels = F.interpolate(labels.unsqueeze(1).float(), size=(256, 256), mode='nearest').squeeze(1).long()
      # Zero the parameter gradients
      optimizer.zero_grad()

      # Forward pass
      outputs = model(images)

      # Calculate the loss
      loss = criterion(outputs, labels)

      # Backward pass and optimize
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete.")


Epoch [1/10], Loss: 0.0550
Epoch [2/10], Loss: 0.0000
Epoch [3/10], Loss: 0.0000
Epoch [4/10], Loss: 0.0000
Epoch [5/10], Loss: 0.0000
Epoch [6/10], Loss: 0.0000
Epoch [7/10], Loss: 0.0000
Epoch [8/10], Loss: 0.0000
Epoch [9/10], Loss: 0.0000
Epoch [10/10], Loss: 0.0000
Training complete.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Set the model to evaluation mode
model.eval()

# Initialize lists to store ground truth labels and predictions
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:  # Assuming you have a test_loader
        images = images.to(device)
        labels = labels.to(device)
        if len(labels.shape) == 4:
            labels = torch.squeeze(labels, dim=1)

        # Forward pass
        outputs = model(images)

        # Get the predicted class for each pixel
        _, preds = torch.max(outputs, 1)

        #convert labels to integer class labels
        labels = labels.long()
        labels = F.interpolate(labels.unsqueeze(1).float(), size=(256, 256), mode='nearest').squeeze(1).long()

        # Flatten the labels and predictions for metric computation
        all_labels.extend(labels.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())


# Calculate evaluation metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
